# Лабораторная работа 3. Градиентный спуск, обусловленность и регуляризация

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 2 |
| Опора на лекции | лекция 2: градиентный спуск (опр. 2.1) и условие сходимости $0<\eta<1/\lambda_{\max}$ (утв. 2.2), мультиколлинеарность и неустойчивость решения (утв. 2.5), гребневая регрессия (опр. 2.7, теорема 2.9), LASSO (опр. 2.12) |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Увидеть, что граница шага из утверждения 2.2 — не запас прочности, а точная граница; понять, почему плохая обусловленность делает задачу дорогой, а решение неустойчивым; разобраться, что именно делают ridge и LASSO и когда они вообще нужны.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab03_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

> **О нормировке.** В §1 лекции 2 функционал берётся без усреднения:
> $Q(\theta) = \|X\theta - y\|^2$, поэтому $\nabla Q = 2X^{\mathsf T}(X\theta - y)$
> и граница шага равна $1/\lambda_{\max}$. Если делить на $\ell$, граница
> изменится в $\ell$ раз. **Всегда проверяйте нормировку своего $Q$** — это самая
> частая причина «почему у меня градиентный спуск разошёлся».

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.linear_model import Lasso, LinearRegression, Ridge
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=3)
describe_variant(variant)

---
# Часть 1. Граница шага — точная

Определение 2.1: $\theta^{(t+1)} = \theta^{(t)} - \eta\,\nabla Q(\theta^{(t)})$.
Утверждение 2.2 обещает сходимость при $0 < \eta < 1/\lambda_{\max}$, где
$\lambda_{\max}$ — наибольшее собственное число $X^{\mathsf T}X$.

Идея доказательства подсказывает, что проверять: ошибка в собственном базисе
умножается на $|1 - 2\eta\lambda_i|$. При $\eta = 1/\lambda_{\max}$ множитель
равен ровно единице — то есть граница должна быть **точной**, а не с запасом.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def grad_Q(theta, X, y):
    """Градиент Q(theta) = ||X theta - y||^2.

    Формула выведена в лекции 1, §6: 2 X^T (X theta - y).
    """
    # TODO (1 строка)
    raise NotImplementedError

In [ ]:
def gradient_descent(X, y, eta, n_iter=120):
    """Траектория метода из определения 2.1."""
    theta = np.zeros(X.shape[1])
    traj = [theta.copy()]
    for _ in range(n_iter):
        theta = theta - eta * grad_Q(theta, X, y)
        traj.append(theta.copy())
    return np.array(traj)


n = 200
x1 = rng.normal(0, 1, n)
X_gd = np.column_stack([x1, 0.6 * x1 + 0.8 * rng.normal(0, 1, n)])
y_gd = X_gd @ np.array([2.0, -1.0]) + rng.normal(0, 0.5, n)

theta_star = np.linalg.lstsq(X_gd, y_gd, rcond=None)[0]
lam_max = np.linalg.eigvalsh(X_gd.T @ X_gd).max()
print(f"theta* = {np.round(theta_star, 4)}")
print(f"lambda_max = {lam_max:.1f}, граница шага 1/lambda_max = {1 / lam_max:.2e}")

In [ ]:
Q = lambda th: np.sum((X_gd @ th - y_gd) ** 2)
Q_star = Q(theta_star)

fig, ax = plt.subplots()
for c in [0.1, 0.5, 0.9, 1.0, 1.05]:
    gap = np.array([Q(th) for th in gradient_descent(X_gd, y_gd, c / lam_max)]) - Q_star
    ax.semilogy(np.where(np.isfinite(gap) & (gap > 0), gap, np.nan), lw=1.8,
                label=fr"$\eta = {c}/\lambda_{{max}}$")
ax.set_xlabel("итерация"); ax.set_ylabel(r"$Q(\theta^{(t)}) - Q(\theta^*)$")
ax.set_title("Сходимость и граница утверждения 2.2"); ax.legend()
plt.tight_layout(); plt.show()

> **Вывод.** При каком множителе спуск перестал сходиться? Совпало ли это с границей утверждения 2.2? Как выглядит траектория при шаге у самой границы?
>
> *(ваш ответ здесь)*

---
# Часть 2. Цена плохой обусловленности

При оптимальном постоянном шаге ошибка убывает как
$\bigl(\frac{\kappa-1}{\kappa+1}\bigr)^t$, где
$\kappa = \mathrm{cond}(X^{\mathsf T}X) = \lambda_{\max}/\lambda_{\min}$.
Значит, число итераций до заданной точности растёт **линейно по $\kappa$**.

In [ ]:
def design_with_cond(n_obj, p, kappa, generator):
    """Матрица X с заданным cond(X^T X) = kappa."""
    A = generator.normal(size=(n_obj, p))
    U, _, Vt = np.linalg.svd(A, full_matrices=False)
    return U @ np.diag(np.logspace(0, -0.5 * np.log10(kappa), p)) @ Vt


def iters_to_converge(X, y, tol=1e-6, max_iter=400_000):
    ev = np.linalg.eigvalsh(X.T @ X)
    eta = 1.0 / (ev.max() + ev.min())              # оптимальный постоянный шаг
    opt = np.linalg.lstsq(X, y, rcond=None)[0]
    theta = np.zeros(X.shape[1])
    for t in range(1, max_iter + 1):
        theta = theta - eta * grad_Q(theta, X, y)
        if np.linalg.norm(theta - opt) < tol * np.linalg.norm(opt):
            return t
    return np.nan

In [ ]:
gen = np.random.default_rng(RANDOM_STATE)
rows = []
for kappa in np.logspace(1, 4.5, 8):
    Xk = design_with_cond(300, 8, kappa, gen)
    yk = Xk @ gen.normal(size=8) + gen.normal(0, 0.01, 300)
    rows.append({"kappa": np.linalg.cond(Xk.T @ Xk), "итераций": iters_to_converge(Xk, yk)})
tab = pd.DataFrame(rows)

fig, ax = plt.subplots()
ax.loglog(tab["kappa"], tab["итераций"], "o-", lw=2, label="эксперимент")
ax.loglog(tab["kappa"], tab["kappa"] * np.log(1e6) / 2, "k--", label=r"$\propto \kappa$")
ax.set_xlabel(r"$\kappa = \mathrm{cond}(X^{\mathsf{T}}X)$")
ax.set_ylabel(r"итераций до $10^{-6}$"); ax.legend()
ax.set_title("Цена плохой обусловленности")
plt.tight_layout(); plt.show()
display(tab.round(0))

> **Вывод.** Как число итераций зависит от $\kappa$? Что это означает для признаков, измеренных в разных единицах?
>
> *(ваш ответ здесь)*

---
# Часть 3. Мультиколлинеарность: решение перестаёт быть определённым

Признаки мультиколлинеарны, если столбцы $X$ почти линейно зависимы. Тогда у
$X^{\mathsf T}X$ есть собственное число, близкое к нулю, и по утверждению 2.5
малое возмущение данных даёт большое изменение решения:

$$
\frac{\|\delta\theta\|}{\|\theta\|} \le \mathrm{cond}(X^{\mathsf T}X)\,
\frac{\|\delta b\|}{\|b\|}.
$$

Посмотрим на это не через оценку, а напрямую: обучим модель на 300
бутстреп-выборках и нарисуем облако полученных решений.

In [ ]:
rho = 0.992                                   # корреляция признаков
z = rng.normal(size=200)
X_col = np.column_stack([z, rho * z + np.sqrt(1 - rho ** 2) * rng.normal(size=200)])
y_col = X_col @ np.array([1.0, 1.0]) + rng.normal(0, 0.3, 200)

print(f"корреляция признаков: {np.corrcoef(X_col.T)[0, 1]:.3f}")
print(f"cond(X^T X) = {np.linalg.cond(X_col.T @ X_col):.1f}")
print(f"МНК-решение = {np.round(np.linalg.lstsq(X_col, y_col, rcond=None)[0], 3)}"
      f"   (истина [1. 1.])")

In [ ]:
boots = np.array([np.linalg.lstsq(X_col[idx], y_col[idx], rcond=None)[0]
                  for idx in (rng.integers(0, 200, 200) for _ in range(300))])

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(boots[:, 0], boots[:, 1], s=10, alpha=0.4, color="#128C7E")
ax.plot(1, 1, "*", ms=16, color="black", label="истина")
ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$"); ax.legend()
ax.set_title(fr"300 бутстреп-оценок при $\rho = {rho}$")
plt.tight_layout(); plt.show()

print(f"ст. отклонение theta_1:              {boots[:, 0].std():.3f}")
print(f"ст. отклонение суммы theta_1+theta_2: {boots.sum(axis=1).std():.3f}")
print(f"корреляция оценок:                   {np.corrcoef(boots.T)[0, 1]:.3f}")

> **Вывод.** Вдоль какого направления вытянуто облако и почему? Какая величина оценивается устойчиво, а какая — нет?
>
> *(ваш ответ здесь)*

---
# Часть 4. Ridge и LASSO: что они делают с решением

Определение 2.7 и теорема 2.9: гребневая регрессия минимизирует
$\|X\theta-y\|^2 + \lambda\sum_{j\ge1}\theta_j^2$, и решение равно
$(X^{\mathsf T}X + \lambda D)^{-1}X^{\mathsf T}y$, где $D$ не штрафует свободный
член. Матрица $X^{\mathsf T}X+\lambda D$ положительно определена при любом
$\lambda>0$ — **даже если $X^{\mathsf T}X$ вырождена**.

LASSO (опр. 2.12) штрафует $\sum|\theta_j|$. Явной формулы нет, но поведение
качественно иное: решение становится **разреженным**. Посмотрим на
регуляризационные пути.

In [ ]:
# Разреженная задача: из 12 признаков полезны ровно три
n_obj = 120
X_sp = rng.normal(size=(n_obj, 12))
theta_sparse = np.zeros(12); theta_sparse[[0, 3, 7]] = [3.0, -2.0, 1.5]
y_sp = X_sp @ theta_sparse + rng.normal(0, 0.3, n_obj)

alphas = np.logspace(-3, 1.2, 60)
path_ridge = np.array([Ridge(alpha=a).fit(X_sp, y_sp).coef_ for a in alphas])
path_lasso = np.array([Lasso(alpha=a, max_iter=50_000).fit(X_sp, y_sp).coef_
                       for a in alphas])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, path, name in [(axes[0], path_ridge, "Ridge ($L_2$)"),
                       (axes[1], path_lasso, "LASSO ($L_1$)")]:
    for j in range(12):
        useful = j in (0, 3, 7)
        ax.plot(alphas, path[:, j], lw=2.2 if useful else 1.0,
                color=None if useful else "grey", alpha=1.0 if useful else 0.5)
    ax.set_xscale("log"); ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$\theta_j$"); ax.set_title(name)

axes[2].semilogx(alphas, (np.abs(path_ridge) > 1e-8).sum(axis=1), lw=2, label="Ridge")
axes[2].semilogx(alphas, (np.abs(path_lasso) > 1e-8).sum(axis=1), lw=2, label="LASSO")
axes[2].axhline(3, ls="--", color="black", label="истинное число ненулевых")
axes[2].set_xlabel(r"$\alpha$"); axes[2].set_ylabel("ненулевых коэффициентов")
axes[2].set_title("Разреженность"); axes[2].legend()
plt.tight_layout(); plt.show()
print("серым — признаки, истинный коэффициент которых равен нулю")

### Задание 4.1. Геометрия: почему $L_1$ обнуляет

Линии уровня $Q$ — эллипсы. Рост $\lambda$ равносилен поиску минимума $Q$ на
множестве $\{\|\theta\|_1 \le c\}$ (ромб) или $\{\|\theta\|_2 \le c\}$ (круг).
Найдите этот минимум перебором точек границы и посмотрите, куда он попадает.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

Xg = np.array([[1.0, 0.35], [0.35, 1.0], [0.5, 0.7]])
yg = np.array([1.4, 0.6, 1.0])
Qg = lambda t: np.sum((Xg @ t - yg) ** 2)
c = 0.6

ts = np.linspace(0, 1, 2001)
border_l1 = np.vstack([np.column_stack([c * (1 - ts), c * ts]),
                       np.column_stack([-c * ts, c * (1 - ts)]),
                       np.column_stack([-c * (1 - ts), -c * ts]),
                       np.column_stack([c * ts, -c * (1 - ts)])])
phi = np.linspace(0, 2 * np.pi, 4001)
border_l2 = np.column_stack([c * np.cos(phi), c * np.sin(phi)])

# TODO (2-3 строки): для каждой границы найдите точку с минимальным Qg
#                    и напечатайте её координаты.

> **Вывод.** Куда попал минимум для ромба и куда — для круга? Почему $L_1$ обнуляет координаты, а $L_2$ — нет?
>
> *(ваш ответ здесь)*

---
# Часть 5. Когда регуляризация вообще нужна

Часто пропускаемый вопрос. Проверим на индивидуальной выборке в двух режимах:
как есть и после расширения признакового пространства до $p > \ell$.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

data = load_personal(variant)
Xtr_full, Xte = data["X_train"], data["X_test"]
ytr_full, yte = data["y_train"], data["y_test"]
mse = lambda pred: np.mean((pred - yte) ** 2)

print(f"объектов {Xtr_full.shape[0]}, признаков {Xtr_full.shape[1]}")
print(f"МНК:       MSE = {mse(LinearRegression().fit(Xtr_full, ytr_full).predict(Xte)):.4g}")
print(f"Ridge(1):  MSE = {mse(Ridge(alpha=1.0).fit(Xtr_full, ytr_full).predict(Xte)):.4g}")
print(f"константа: MSE = {mse(ytr_full.mean()):.4g}")

In [ ]:
# Теперь p > l: полиномиальное расширение и урезанная обучающая выборка
poly, scaler = PolynomialFeatures(degree=2, include_bias=False), StandardScaler()
Xs_tr = scaler.fit_transform(poly.fit_transform(Xtr_full[:120]))
Xs_te = scaler.transform(poly.transform(Xte))
ys_tr = ytr_full[:120]
print(f"объектов {Xs_tr.shape[0]}, признаков {Xs_tr.shape[1]} -> p > l")

Xa, Xv, ya, yv = train_test_split(Xs_tr, ys_tr, test_size=0.3, random_state=RANDOM_STATE)
alpha_max = np.abs(Xa.T @ (ya - ya.mean())).max() / len(ya)
grid = np.logspace(np.log10(alpha_max) - 4, np.log10(alpha_max), 30)

### Задание 5.1. Подбор $\alpha$ по отложенной части

Контрольную выборку трогать нельзя — она нужна для финальной оценки. Поэтому
$\alpha$ подбираем по отложенной части обучающей. Полноценный скользящий
контроль будет на занятии 5.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

rows = []
for name, Model in [("Ridge", Ridge), ("LASSO", Lasso)]:
    # TODO (3-4 строки): по сетке grid обучите Model(alpha=a) на (Xa, ya),
    #   выберите a_best по MSE на (Xv, yv), переобучите на всей (Xs_tr, ys_tr),
    #   запишите MSE на контроле и число ненулевых коэффициентов.
    ...

theta_ols = np.linalg.lstsq(Xs_tr, ys_tr - ys_tr.mean(), rcond=None)[0]
print(f"МНК:       MSE = {mse(Xs_te @ theta_ols + ys_tr.mean()):.4g}")
print(f"константа: MSE = {mse(ys_tr.mean()):.4g}")

> **Вывод.** В каком из двух режимов регуляризация дала выигрыш и почему? Сформулируйте правило, когда она нужна.
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Утверждение 2.2 требует $\eta < 1/\lambda_{\max}$. Как изменится граница, если минимизировать усреднённый функционал $\frac1\ell\|X\theta-y\|^2$?
2. Два признака имеют корреляцию 0.999. Что произойдёт с оценками их коэффициентов и с их суммой? Какая величина оценивается устойчиво?
3. Почему в определении 2.7 свободный член не штрафуется? Что произойдёт с решением, если прибавить ко всем $y_i$ константу 1000, а $\theta_0$ при этом штрафовать?
4. Ridge и LASSO дали одинаковое качество. Какой выберете и почему? Приведите по одному аргументу за каждый.

---

**Дома:** откройте `lab03_homework.ipynb` — там три задачи: свой SGD с расписаниями шага, LASSO покоординатным спуском и байесовский взгляд на регуляризацию.